# Step 3 — Results and publication figures

Reads the posterior chains (`.npz` + `_meta.json`) written by step 2 and renders the full suite
of publication figures with [`photoring.plotting`](photoring/plotting.py): posterior-predictive
checks, marginals with prior overlays, corner plots, the projected ring-geometry diagram, a
consolidated results panel, and summary tables.

All figures are written under `pipeline/<CASE>/figures/<type>/`.

## 0. Environment and style

In [ ]:
# ── Bootstrap: make the sibling packages importable without installation ────
# The pipeline uses three packages that live in the repository, uninstalled:
#   exorings, geotrans   (repo root)      photoring   (pipeline/)
# We locate the repo root and pipeline/ robustly from the current working dir
# (Jupyter / nbconvert / papermill all run notebooks from pipeline/).
import sys, pathlib
_HERE   = pathlib.Path.cwd().resolve()
_cands  = [_HERE, *_HERE.parents]
_NB_DIR = next((c for c in _cands if (c / "photoring").is_dir()), _HERE)
_REPO   = next((c for c in _cands if (c / "exorings").is_dir()), _NB_DIR.parent)
for _p in (str(_REPO), str(_NB_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root :", _REPO)
print("pipeline  :", _NB_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import photoring as pr
import photoring.plotting as plot
from photoring.io import load_observables

# Edit STYLE overrides here (e.g. use_latex=True needs a TeX install).
plot.apply_style({"use_latex": False})

## 1. Configuration  ← point at your case / runs

In [ ]:
CASE          = "kepler_51"
FORWARD_MODEL = "exorings"          # which results/<model>/ directory to read
RUN_TAGS      = []                   # [] -> auto-discover all runs in that directory
PPC_OBSERVABLES = ["delta", "T14", "T23", "rho_obs", "b_obs"]
PLANETS       = ["b", "d"]           # planets whose TTV observables to overlay

paths = pr.CasePaths(CASE)
paths.ensure_outputs(FORWARD_MODEL)
print("results dir:", paths.results_dir(FORWARD_MODEL))

## 2. Load runs and TTV observables

In [ ]:
RUNS = pr.discover_runs(paths.results_dir(FORWARD_MODEL), RUN_TAGS or None)
RUNS_LIST = list(RUNS.values())
print(f"loaded {len(RUNS_LIST)} run(s)")
for r in RUNS_LIST:
    print(f"  {r['tag']}  planet={r['planet']}  lnZ={r['logz']:.2f}  N={len(r['chain'])}")

TTV = {pl: load_observables(paths.observables_file(pl)) for pl in PLANETS
       if paths.observables_file(pl).exists()}
try:
    BERGER = np.loadtxt(paths.rho_true_samples) / 1000.0
except Exception:
    BERGER = None

## 3. Per-run figures

In [ ]:
for r in RUNS_LIST:
    print(f"\n== {r['tag']} ==")
    ttv = TTV.get(r["planet"], {})
    if ttv:
        plot.plot_ppc(r, ttv, obs_keys=PPC_OBSERVABLES, paths=paths); plt.show()
    plot.plot_marginals(r, berger_rho=BERGER, paths=paths); plt.show()
    plot.plot_corner(r, paths=paths); plt.show()
    try:
        plot.plot_ring_diagram(r, paths=paths); plt.show()
    except Exception as e:
        print("  ring diagram skipped:", e)

## 4. Consolidated results panel (per run)

In [ ]:
for r in RUNS_LIST:
    ttv = TTV.get(r["planet"], {})
    if not ttv:
        continue
    try:
        plot.plot_results_panel(r, ttv, obs_keys=["delta", "rho_obs", "T14", "b_obs"], paths=paths)
        plt.show()
    except Exception as e:
        print(f"  panel skipped for {r['tag']}:", e)

## 5. Summary tables

In [ ]:
if RUNS_LIST:
    plot.print_summary_table(RUNS_LIST)
    print()
    plot.print_latex_table(RUNS_LIST)
else:
    print("No runs loaded — run step 2 first.")